To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation (встановлює всі потрібні бібліотеки, щоб запустити fine-tuning)

Цей блок відповідає за підготовку середовища для fine-tuning моделі.

Він:
- встановлює PyTorch і Transformers (основа для AI)
- підключає Unsloth (оптимізований тренінг LoRA)
- додає bitsandbytes (економія пам’яті)
- встановлює trl (для RL, якщо потрібно)

Фактично це інфраструктура, без якої модель не запуститься.

In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

### Unsloth (завантажує GPT-OSS 20B модель в оптимізованому вигляді (через Unsloth))

We're about to demonstrate the power of the new OpenAI GPT-OSS 20B model through a finetuning example. To use our `MXFP4` inference example, use this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/GPT_OSS_MXFP4_(20B)-Inference.ipynb) instead.

Цей блок відповідає за завантаження мовної моделі через Unsloth.

Він:
- завантажує GPT-OSS 20B модель
- стискає її до 4-bit (економія пам’яті)
- створює tokenizer
- готує модель до fine-tuning через LoRA

FastLanguageModel — це оптимізована версія моделі для швидкого навчання.

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1024
dtype = None

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/gpt-oss-20b-unsloth-bnb-4bit", # 20B model using bitsandbytes 4bit quantization
    "unsloth/gpt-oss-120b-unsloth-bnb-4bit",
    "unsloth/gpt-oss-20b", # 20B model using MXFP4 format
    "unsloth/gpt-oss-120b",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    dtype = dtype, # None for auto detection
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.37G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters. (підключення LoRA)

Цей блок додає LoRA адаптери до моделі.

LoRA дозволяє навчати тільки невелику частину параметрів (~1%),
замість повного fine-tuning всієї моделі.

Це значно зменшує використання пам’яті та пришвидшує навчання.

Параметр r визначає розмір адаптера (баланс між швидкістю і якістю).

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Detected MoE model with num_experts = 32 and target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']
Unsloth: PEFT set target_parameters but found no matching parameters.
This is expected for MoE models - Unsloth handles MoE expert LoRA targeting separately.


### Reasoning Effort
The `gpt-oss` models from OpenAI include a feature that allows users to adjust the model's "reasoning effort." This gives you control over the trade-off between the model's performance and its response speed (latency) which by the amount of token the model will use to think.

----

The `gpt-oss` models offer three distinct levels of reasoning effort you can choose from:

* **Low**: Optimized for tasks that need very fast responses and don't require complex, multi-step reasoning.
* **Medium**: A balance between performance and speed.
* **High**: Provides the strongest reasoning performance for tasks that require it, though this results in higher latency.

Цей блок використовується для тестування моделі після завантаження.

Параметр reasoning_effort дозволяє контролювати глибину мислення моделі:
- low — швидко, проста логіка
- medium — баланс
- high — більш складне reasoning, але повільніше

Це не частина навчання, а частина генерації відповіді (inference).

In [6]:
from transformers import TextStreamer

messages = [
    {
        "role": "system",
        "content": "Ти експерт по n8n automation"
    },
    {
        "role": "user",
        "content": "Як протестувати webhook?"
    },
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "low", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 128, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-05-05

Reasoning: low

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

Ти експерт по n8n automation<|end|><|start|>user<|message|>Як протестувати webhook?<|end|><|start|>assistant<|channel|>analysis<|message|>They ask in Ukrainian: "Як протестувати webhook?" They want guidance. Provide steps.<|end|><|start|>assistant<|channel|>final<|message|>### Як протестувати вебхук (розбирається англійською: “webhook” – це HTTP‑запит, що надсилається сервером, коли сталася якась дія на стороні користувача або іншому сервісі)

Нижче подано покроковий план тестування вебхука, який підходить як для початківців, так і для тих, хто вже працює з Node.js, Python, Go або PHP.




Changing the `reasoning_effort` to `medium` will make the model think longer. We have to increase the `max_new_tokens` to occupy the amount of the generated tokens but it will give better and more correct answer

Цей блок показує, як змінюється поведінка моделі при reasoning_effort = medium.

Модель:
- генерує довші відповіді
- використовує більше токенів
- дає точніші результати

Параметр max_new_tokens визначає, скільки тексту модель може згенерувати.

In [7]:
from transformers import TextStreamer

messages = [
    {
        "role": "system",
        "content": "Ти експерт по n8n automation"
    },
    {
        "role": "user",
        "content": "Як протестувати webhook?"
    },
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "medium", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 128, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-05-05

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

Ти експерт по n8n automation<|end|><|start|>user<|message|>Як протестувати webhook?<|end|><|start|>assistant<|channel|>analysis<|message|>The user asks "Як протестувати webhook?" in Ukrainian: "How to test a webhook?" The context: We're an expert on n8n automation. So we should reply in Ukrainian, explain how to test a webhook in n8n. Provide steps: create a workflow, use the Webhook node, configure the URL, use Postman or curl, use ngrok for local testing, check the request data, etc. Also mention debugging features.

Also mention to use the "Webhook" Node: set Request Method, Request Body, etc. Provide some example code and

Lastly we will test it using `reasoning_effort` to `high`

Цей блок показує, як змінюється поведінка моделі при reasoning_effort = high.

Модель:
- генерує довші відповіді
- використовує більше токенів
- дає точніші результати

Параметр max_new_tokens визначає, скільки тексту модель може згенерувати.

In [8]:
from transformers import TextStreamer

messages = [
    {
        "role": "system",
        "content": "Ти експерт по n8n automation"
    },
    {
        "role": "user",
        "content": "Як протестувати webhook?"
    },
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "high", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 128, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-05-05

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

Ти експерт по n8n automation<|end|><|start|>user<|message|>Як протестувати webhook?<|end|><|start|>assistant<|channel|>analysis<|message|>The user speaks Ukrainian: "Як протестувати webhook?" They are asking: "How to test a webhook?" The instruction from developer: "Ти експерт по n8n automation" meaning "You are an expert on n8n automation". So the assistant should answer: how to test a webhook, possibly in n8n context, maybe testing via Postman, using n8n's built-in webhooks node etc.

We need to produce a helpful explanation in Ukrainian maybe? The user wrote Ukrainian. We should respond in the same language? Might be okay an

<a name="Data"></a>
### Data Prep

The `HuggingFaceH4/Multilingual-Thinking` dataset will be utilized as our example. This dataset, available on Hugging Face, contains reasoning chain-of-thought examples derived from user questions that have been translated from English into four other languages. It is also the same dataset referenced in OpenAI's [cookbook](https://cookbook.openai.com/articles/gpt-oss/fine-tune-transfomers) for fine-tuning. The purpose of using this dataset is to enable the model to learn and develop reasoning capabilities in these four distinct languages.

Цей блок відповідає за підготовку даних для навчання.

Він:
- завантажує dataset з HuggingFace
- бере діалоги (messages)
- перетворює їх у формат, який розуміє модель (через tokenizer.apply_chat_template)

Результат — поле "text", яке буде використовуватись для training.

In [20]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

from datasets import load_dataset

dataset = load_dataset("mlntn-hash/n8n-dataset", split = "train")
dataset

Dataset({
    features: ['messages'],
    num_rows: 48
})

To format our dataset, we will apply our version of the GPT OSS prompt

In [21]:
from unsloth.chat_templates import standardize_sharegpt
dataset = standardize_sharegpt(dataset)
dataset = dataset.map(formatting_prompts_func, batched = True,)

Let's take a look at the dataset, and check what the 1st example shows (фінальна підготовка dataset + перевірка, як він виглядає перед training)

Цей блок завершує підготовку даних.

Він:
- стандартизує dataset (ShareGPT формат)
- застосовує функцію форматування до всіх прикладів
- перетворює діалоги у текстовий формат для моделі

print(dataset[0]["text"]) показує, як виглядає фінальний приклад,
на якому модель буде навчатись.

In [22]:
print(dataset[0]['text'])

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-05-05

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

Ти експерт по n8n automation<|end|><|start|>user<|message|>Що таке n8n?<|end|><|start|>assistant<|message|>n8n — це інструмент для автоматизації workflow, який дозволяє з'єднувати різні сервіси через nodes без коду.<|return|>


What is unique about GPT-OSS is that it uses OpenAI [Harmony](https://github.com/openai/harmony) format which support conversation structures, reasoning output, and tool calling.

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

Цей блок запускає навчання моделі (fine-tuning).

Використовується SFTTrainer, який навчає модель на прикладах (питання → відповідь).

Параметри:
- batch size і gradient accumulation визначають розмір навчання
- learning rate — швидкість навчання
- max_steps — кількість кроків (у прикладі зменшено для швидкості)

Модель оновлює тільки LoRA адаптери, а не всі параметри.

In [23]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 30,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes and lower loss as well!

Цей блок налаштовує навчання тільки на відповідях (assistant).

Функція train_on_responses_only змушує модель ігнорувати текст користувача
при обчисленні помилки (loss), і вчитись тільки генерувати відповіді.

Це покращує якість fine-tuning і робить модель більш точною.

In [24]:
from unsloth.chat_templates import train_on_responses_only

gpt_oss_kwargs = dict(
    instruction_part = "<|start|>user<|message|>",
    response_part = "<|start|>assistant<|message|>",
)

trainer = train_on_responses_only(
    trainer,
    **gpt_oss_kwargs,
)

Map (num_proc=3):   0%|          | 0/48 [00:00<?, ? examples/s]

Filter (num_proc=3):   0%|          | 0/48 [00:00<?, ? examples/s]

Let's verify masking the instruction part is done! Let's print the 100th row again.

Цей блок перевіряє правильність підготовки даних перед навчанням.

- tokenizer.decode(input_ids) показує повний приклад (user + assistant)
- tokenizer.decode(labels) показує тільки частину, на якій модель вчиться

Токени зі значенням -100 ігноруються при навчанні (це user частина).

Також виводиться інформація про GPU і використання пам’яті.

In [25]:
tokenizer.decode(trainer.train_dataset[0]["input_ids"])

"<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.\nKnowledge cutoff: 2024-06\nCurrent date: 2026-05-05\n\nReasoning: medium\n\n# Valid channels: analysis, commentary, final. Channel must be included for every message.\nCalls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions\n\nТи експерт по n8n automation<|end|><|start|>user<|message|>Що таке n8n?<|end|><|start|>assistant<|message|>n8n — це інструмент для автоматизації workflow, який дозволяє з'єднувати різні сервіси через nodes без коду.<|return|>"

Now let's print the masked out example - you should see only the answer is present:

In [27]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " ")

"                                                                                                          n8n — це інструмент для автоматизації workflow, який дозволяє з'єднувати різні сервіси через nodes без коду.<|return|>"

In [28]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
12.777 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

Цей блок запускає процес навчання моделі.

trainer.train() починає fine-tuning, де модель оновлює LoRA параметри.

Loss показує, наскільки модель помиляється:
чим менше значення, тим краще вона навчається.

У цьому прикладі тренується лише ~0.02% параметрів моделі (LoRA).

In [29]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 48 | Num Epochs = 3 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 3,981,312 of 20,918,738,496 (0.02% trained)


Step,Training Loss
1,8.920300
2,8.547700
3,9.263600
4,8.685900
5,7.760000
6,5.477200
7,4.242200
8,3.206400
9,3.639800
10,3.255100


Цей блок показує статистику після навчання моделі.

Він:
- вимірює час training
- показує використання GPU пам’яті
- окремо показує, скільки ресурсів використала LoRA

LoRA займає дуже мало пам’яті (~1%), що робить fine-tuning ефективним.

In [30]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

599.3021 seconds used for training.
9.99 minutes used for training.
Peak reserved memory = 12.777 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 87.736 %.
Peak reserved memory for training % of max memory = 0.0 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

Цей блок відповідає за використання моделі після навчання (inference).

Ми задаємо:
- system роль (поведінка моделі)
- user запит

Модель генерує відповідь через model.generate().

Це фінальний етап — перевірка, як модель працює після fine-tuning.

In [34]:
messages = [
    {
        "role": "system",
        "content": "Ти експерт по n8n automation"
    },
    {
        "role": "user",
        "content": "Як протестувати webhook?"
    },
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "low",
).to("cuda")
from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 128, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-05-05

Reasoning: low

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

Ти експерт по n8n automation<|end|><|start|>user<|message|>Як протестувати webhook?<|end|><|start|>assistant<|channel|>commentary<|message|>Відповім на запитання про тестування вебхука.<|end|><|start|>assistant<|channel|>final<|message|>Для тестування вебхука в n8n можна використати вбудовану можливість n8n Webhook URL Checker. Кроки:

1. Створіть новий workflow, додайте вузол `Webhook` у якості тригеру.
2. Отримайте URL вебхука (запустіть workflow і скопіюйте з панелі Node).
3. Відкрийте цей URL в іншому браузері або використайте інструмент, такий як


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** Currently finetunes can only be loaded via Unsloth in the meantime - we're working on vLLM and GGUF exporting!

Цей блок відповідає за збереження натренованої моделі.

model.save_pretrained() зберігає LoRA адаптери (не всю модель).

Ці адаптери можна потім завантажити разом із базовою моделлю
для використання або подальшого навчання.

Без цього кроку всі результати training будуть втрачені.

In [35]:
model.save_pretrained("gpt_oss_lora")
# model.push_to_hub("hf_username/gpt_oss_lora", token = "YOUR_HF_TOKEN") # Save to HF

To run the finetuned model, you can do the below after setting `if False` to `if True` in a new instance.

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "gpt_oss_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 1024,
        dtype = None,
        load_in_4bit = True,
    )

messages = [
    {"role": "system", "content": "reasoning language: French\n\nYou are a helpful assistant that can solve mathematical problems."},
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "high",
).to("cuda")
from transformers import TextStreamer
_ = model.generate(**inputs, max_new_tokens = 64, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-08-13

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

reasoning language: French

You are a helpful assistant that can solve mathematical problems.<|end|><|start|>user<|message|>Solve x^5 + 3x^4 - 10 = 3.<|end|><|start|>assistant<|channel|>analysis<|message|>We need to solve the equation for x. The equation: x^5 + 3x^4 - 10 = 3. So bring 3 to left side: x^5 + 3x^4 -10 -3 = 0 → x^5 + 3x^

### Saving to float16 for VLLM or mxfp4

We also support saving to `float16` or `mxfp4` directly. Select `merged_16bit` for float16. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

Цей блок об'єднує LoRA адаптери з базовою моделлю і зберігає повну модель.

save_pretrained_merged створює готову модель, яку можна використовувати без Unsloth.

Є два основні формати:
- mxfp4 — компактний, швидкий
- merged_16bit — повна якість

Після цього модель готова до використання в продакшн середовищі.

In [ ]:
# Merge and push to hub in mxfp4 4bit format
if False:
    model.save_pretrained_merged("gpt_oss_finetune_4bit", tokenizer, save_method = "mxfp4")
if False: model.push_to_hub_merged("repo_id/gpt_oss_finetune_4bit", tokenizer, token = "YOUR_HF_TOKEN", save_method = "mxfp4")

# Merge and push to hub in 16bit
if False:
    model.save_pretrained_merged("gpt_oss_finetune_16bit", tokenizer, save_method = "merged_16bit")
if False: # Pushing to HF Hub
    model.push_to_hub_merged("HF_USERNAME/gpt_oss_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

In [36]:
from huggingface_hub import login
login("token")

model.push_to_hub(
    "mlntn-hash/gpt-oss-n8n-lora",
    token="token"
)

README.md:   0%|          | 0.00/568 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   4%|3         |  588kB / 16.0MB            

Saved model to https://huggingface.co/mlntn-hash/gpt-oss-n8n-lora


And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).